<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# ARIMA and SARIMA

**[ARIMA and SARIMA](https://learning.nextia-ai.com/courses/forecasting/m04/arima-and-sarima/)** · Forecasting and Time Series: Predict What Comes Next · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute differences by hand, let AutoARIMA choose a model for the parasol, read its forecast range as it widens with the horizon, see what goes wrong without a season, and compare AutoARIMA with ETS in a small backtest.

| | |
|---|---|
| **Time** | About 40 minutes, of which about 100 seconds of waiting for the small backtest |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | ETS and the backtest table, from [Exponential smoothing (ETS, Holt-Winters)](https://learning.nextia-ai.com/courses/forecasting/m04/exponential-smoothing/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m04/arima-and-sarima/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M04-L02-arima-and-sarima/arima-and-sarima-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Differences: the change from one day to the next

ARIMA does not model the sales themselves. It often models the **differences**: today's value minus yesterday's. A series that climbs becomes a series of changes that stay around one level.

> **Predict.** The series is 20, 22, 25, 24, 28, 31, 30. What are its six differences? Then run the cell.

In [ ]:
series = np.array([20, 22, 25, 24, 28, 31, 30])
changes = np.diff(series)
print("differences:", changes.tolist())
print("mean change:", round(changes.mean(), 2))

> **Check.** You should see `differences: [2, 3, -1, 4, 3, -1]` and `mean change: 1.67`.

The series rises from 20 to 30. The differences do not rise: they go up and down around 1.67. A model of the differences forecasts a change, and adds it to the last value. The **I** in ARIMA (integrated) is this step back from changes to values.

## 2. AutoARIMA chooses a model for the parasol

`AutoARIMA` tries many combinations of past values, differences and past errors, with and without a weekly part, and keeps the one with the best fit score (AICc). The next cell fits it to the parasol up to Sunday 22 June 2025 and asks for an 80% and a 90% range.

> **Predict.** The range says where the sales will probably fall. Will it be as wide on day 14 as on day 1?

In [ ]:
from statsforecast import StatsForecast
from statsforecast.arima import arima_string
from statsforecast.models import AutoARIMA

parasol = load_sales(["P025"])
origin = pd.Timestamp("2025-06-22")
train = parasol[parasol["ds"] <= origin][["unique_id", "ds", "y"]]
real = parasol[(parasol["ds"] > origin) & (parasol["ds"] <= origin + pd.Timedelta(days=14))]
sf = StatsForecast(models=[AutoARIMA(season_length=7)], freq="D")
fc = sf.forecast(df=train, h=14, level=[80, 90])
sf.fit(train)
print("model chosen:", arima_string(sf.fitted_[0, 0].model_).strip())
fc["width 80%"] = fc["AutoARIMA-hi-80"] - fc["AutoARIMA-lo-80"]
print(fc[["ds", "AutoARIMA", "AutoARIMA-lo-80", "AutoARIMA-hi-80", "width 80%"]].round(1).iloc[[0, 6, 13]].to_string(index=False))
print("14-day total: AutoARIMA", round(fc["AutoARIMA"].sum(), 1), "| real", int(real["y"].sum()))

> **Check.** You should see `model chosen: ARIMA(0,1,2)(1,0,1)[7]`, three rows (days 1, 7 and 14) with an 80% width of 10.2, 14.1 and 17.7 units, and `14-day total: AutoARIMA 160.1 | real 197`.

The range widens with the horizon: each day's forecast is built on the forecasts of the days before it, so their errors add up. The chart shows the forecast, the two ranges and the real sales.

In [ ]:
import matplotlib.pyplot as plt

past = parasol[(parasol["ds"] > origin - pd.Timedelta(days=42)) & (parasol["ds"] <= origin)]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(past["ds"], past["y"], color="grey", label="sold")
ax.fill_between(fc["ds"], fc["AutoARIMA-lo-90"], fc["AutoARIMA-hi-90"], color="tab:blue", alpha=0.15, label="90% range")
ax.fill_between(fc["ds"], fc["AutoARIMA-lo-80"], fc["AutoARIMA-hi-80"], color="tab:blue", alpha=0.3, label="80% range")
ax.plot(fc["ds"], fc["AutoARIMA"], color="tab:blue", label="AutoARIMA")
ax.plot(real["ds"], real["y"], "o", color="black", label="sold (hidden from the model)")
ax.axvline(origin, color="black", linewidth=1)
ax.set_ylabel("parasols sold")
ax.legend(loc="upper left")
plt.show()

In the heat wave (30 June to 6 July), the last two days are above the 80% range, and the forecast is too low on almost every day. ARIMA, like ETS, knows only the parasol's past.

## 3. Failure case: forget the season

`AutoARIMA()` without `season_length` looks for no weekly part at all: its default season length is 1.

> **Predict.** What will the 14-day forecast look like without a weekly part? Then run the cell.

In [ ]:
sf1 = StatsForecast(models=[AutoARIMA()], freq="D")
fc1 = sf1.forecast(df=train, h=14)
sf1.fit(train)
print("model chosen:", arima_string(sf1.fitted_[0, 0].model_).strip())
print("forecast:", fc1["AutoARIMA"].round(1).tolist())

> **Check.** You should see `model chosen: ARIMA(0,1,2)` and a forecast of about 11.1 units for every day.

No weekly part means a flat line: the busy Saturdays and quiet Wednesdays are gone. Always give `season_length` (7 for daily data with a weekly season).

## 4. ETS against AutoARIMA in a small backtest

The course backtest (50 products, 13 origins) took AutoARIMA about 1,000 seconds. Here we use every fifth product (10 products) and every fourth origin (4 origins, spread over the year): 40 fits for each method.

> **Predict.** On these 10 products, will AutoARIMA have a lower WAPE than ETS? How much longer will it take? Then run the cell.

In [ ]:
from statsforecast.models import AutoETS

def statsforecast_model(model):
    def forecaster(train, h, future):
        sf = StatsForecast(models=[model], freq="D")
        out = sf.forecast(df=train[["unique_id", "ds", "y"]], h=h)
        return out.rename(columns={str(model): "yhat"}).assign(yhat=lambda d: d["yhat"].clip(lower=0))
    return forecaster

few = BACKTEST_PRODUCTS[::5]
sales = load_sales(few)
four = ORIGINS[::4]
fc_ets = bt.backtest(sales, statsforecast_model(AutoETS(season_length=7)), h=14, origins=four)
fc_arima = bt.backtest(sales, statsforecast_model(AutoARIMA(season_length=7)), h=14, origins=four)
fc_snaive = bt.backtest(sales, bt.seasonal_naive(7), h=14, origins=four)
rows = [bt.score(f, sales, name, season=7, cost=bt.stock_cost(PRICES))
        for name, f in [("seasonal naive", fc_snaive), ("ETS (AutoETS)", fc_ets), ("AutoARIMA", fc_arima)]]
table = bt.show(rows)

> **Check.** You should see three rows. ETS has a WAPE % of 35.3 and AutoARIMA 36.5. The fit times are about 4 and 100 seconds.

On 10 products and 4 origins, the numbers move: a small test is noisy. The course table, on all 50 products and 13 origins, has ETS at 41.3% and AutoARIMA at 41.4%, with AutoARIMA more than 21 times slower (996 against 46 seconds).

## 5. Your turn: product by product

Tomás says: "AutoARIMA is the more advanced model, so it wins on most products."

> **Your turn.** For each of the 10 products, add up the absolute errors of `fc_arima` and of `fc_ets` (over all origins and days). Count the products where the AutoARIMA sum is **smaller**. Put the count in `arima_wins` as a Python integer. Then run the check cell.

In [ ]:
arima_wins = ...  # the number of products where AutoARIMA has the smaller absolute error

In [ ]:
expect('arima_wins', arima_wins, 2)

On these 10 products, AutoARIMA wins on 2. In the course backtest, it wins on 19 of the 50 products: fewer than half. The more complex model is not better on most products, and it costs much more time.

## Next

You compared ARIMA with ETS. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m04/arima-and-sarima/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Several seasons and long series](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m04/arima-and-sarima/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.